<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/train_en_azb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers[sentencepiece] sacrebleu -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
!pip install datasets
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

dataset = load_dataset("Kartal-Ol/en-azb-548k")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/548900 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

# Load the Arabic tokenizer
tokenizer = AutoTokenizer.from_pretrained("asafaya/bert-base-arabic")

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/334k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Map:   0%|          | 0/548900 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
from transformers import T5ForConditionalGeneration

# Load T5 model
model = T5ForConditionalGeneration.from_pretrained("t5-small")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          # Where to save model checkpoints
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size per GPU
    per_device_eval_batch_size=16,  # Batch size for validation
    num_train_epochs=5,             # Number of epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_total_limit=3,
    report_to="none" # Keep only the last 3 checkpoints
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [16]:
!pip install wandb


In [11]:
!export WANDB_MODE=disabled


In [ ]:
from transformers import Trainer
import os

# Create the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

# Start training
trainer.train()

# 9. Save
# Save the model and tokenizer to Google Drive
model.save_pretrained('/content/drive/MyDrive/fine_tuned_nmt_model2')
tokenizer.save_pretrained('/content/drive/MyDrive/fine_tuned_nmt_model2')

Epoch,Training Loss,Validation Loss
1,1.829400,3.340679
2,1.692200,3.153324


('/content/drive/MyDrive/fine_tuned_nmt_model/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model/special_tokens_map.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model/vocab.txt',
 '/content/drive/MyDrive/fine_tuned_nmt_model/added_tokens.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
save_path = '/content/drive/MyDrive/fine_tuned_nmt_model2'
os.makedirs(save_path, exist_ok=True)

In [ ]:
dataset['train']['translation'][10]['en']

'The love Christ displayed was central to his accomplishing what God has purposed for mankind .'

First model

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

# Load the fine-tuned model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model")

# Input text to translate
input_text = f"Translate English(en) to South Azerbaijani(azb): how are you today?"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate a translation
output = model.generate(input_ids)
translation = tokenizer.decode(output[0], skip_special_tokens=True)

print("Translation:", translation)


ValueError: could not determine the shape of object type 'torch.storage.UntypedStorage'

In [ ]:
input_text

'Translate English(en) to South Azerbaijani(azb): Yes , He surely is Able to do all things .'

In [ ]:
!ls /content/drive/MyDrive/fine_tuned_nmt_model

config.json		model.safetensors	 tokenizer_config.json	vocab.txt
generation_config.json	special_tokens_map.json  tokenizer.json


## 2 **second** model for machine translation

In [ ]:
from transformers import T5ForConditionalGeneration

# Load T5 model
model = T5ForConditionalGeneration.from_pretrained("t5-base")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.315500,2.578046


Epoch,Training Loss,Validation Loss
1,1.315500,2.578046
2,1.112400,2.284231
3,1.030900,2.166523
4,0.995600,2.114824
5,0.979900,2.097838


('/content/drive/MyDrive/fine_tuned_nmt_model2/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model2/special_tokens_map.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model2/vocab.txt',
 '/content/drive/MyDrive/fine_tuned_nmt_model2/added_tokens.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model2/tokenizer.json')

In [ ]:
# Load the fine-tuned model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model2")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model2")

# Input text to translate
input_text = f"Translate English(en) to Azerbaijani(azb): how are you today?"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate a translation
output = model.generate(input_ids)
translation = tokenizer.decode(output[0], skip_special_tokens=True)

print("Translation:", translation)


Translation: یوخسا سیزین اۆچون یارادیلیشیق


# New way of making machine translation using Ali's idea

In [3]:
def tokenize_function(examples):
    # Tokenize without padding to minimize memory usage
    tokenized = tokenizer(
        examples["text"],
        max_length=512,
        truncation=True
    )
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": tokenized["input_ids"],  # For autoencoding tasks
    }

In [6]:
from datasets import load_dataset

dsval = load_dataset("jalilkartal/alpaca-azerbaijani-arabic-script")
val_dataset = dsval.map(tokenize_function, batched=True)


Map:   0%|          | 0/41601 [00:00<?, ? examples/s]

In [7]:
from datasets import load_dataset

ds = load_dataset("jalilkartal/AZB_EN_Combined_47m_tokenized")

README.md:   0%|          | 0.00/456 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

train-00000-of-00068.parquet:   0%|          | 0.00/414M [00:00<?, ?B/s]

train-00001-of-00068.parquet:   0%|          | 0.00/279M [00:00<?, ?B/s]

...

train-00065-of-00068.parquet:   0%|          | 0.00/182M [00:00<?, ?B/s]

train-00066-of-00068.parquet:   0%|          | 0.00/168M [00:00<?, ?B/s]

train-00067-of-00068.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/184178600 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/69 [00:00<?, ?it/s]

In [5]:
from transformers import T5Tokenizer
# Load the tokenizer from Hugging Face Hub
tokenizer = T5Tokenizer.from_pretrained("jalilkartal/AZB_EN_48m")
# Define the special tokens
special_tokens_dict = {
    "additional_special_tokens": [
        "Translate English to South Azerbaijani:",
        "Translate South Azerbaijani to English:"
    ]
}

# Add special tokens to the tokenizer
num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)
print(f"Added {num_added_tokens} special tokens.")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/893k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/84.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Added 2 special tokens.


In [21]:
from transformers import T5ForConditionalGeneration, T5Config,DataCollatorForSeq2Seq

# Define the model configuration
model_config = T5Config(
    vocab_size=32102,
    n_positions=512,
    d_model=64,
    d_kv=64,
    d_ff=2048,
    num_layers=2,
    num_heads=4,
    relative_attention_num_buckets=32,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-06,
    initializer_factor=1.0,
    is_encoder_decoder=True,
    pad_token_id=0,               # Padding token
    eos_token_id=1,               # End-of-sequence token
    decoder_start_token_id=0      # Decoder start token set to pad_token_id
)

# Initialize the model
model = T5ForConditionalGeneration(config=model_config)



In [27]:
# Define the Data Collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,  # Ensures proper padding
    return_tensors="pt"  # Returns PyTorch tensors
)

In [17]:
import os
os.environ["WANDB_MODE"] = "disabled"
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_DISABLED"] = "true"

In [25]:
print(f"Decoder Start Token ID: {model.config.decoder_start_token_id}")
tokenizer.pad_token_id = 0
tokenizer.eos_token_id = 1
model.config.decoder_start_token_id = model.config.pad_token_id


Decoder Start Token ID: 0


In [28]:
from transformers import Trainer, TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    save_steps=10_000,
    save_total_limit=2,
    fp16=True,
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],  # Tokenized dataset
    eval_dataset=val_dataset["train"],  # Tokenized validation set
    data_collator=data_collator,  # Add the data collator here
    tokenizer=tokenizer,  # Optional: For dynamic padding during evaluation
)

# Start training
trainer.train()
#new_model

# 9. Save
# Save the model and tokenizer to Google Drive
model.save_pretrained('/content/drive/MyDrive/ali_najafi')
tokenizer.save_pretrained('/content/drive/MyDrive/ali_najafi')


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-28-df6f4155c8dc>:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Tokenize function for batched tokenization
def tokenize_function(examples):
    # Extract 'en' and 'azb' for each example in the batch
    source_texts = [example['en'] for example in examples['translation']]
    target_texts = [example['azb'] for example in examples['translation']]

    # Tokenize both source (English) and target (Azerbaijani Arabic script)
    source = tokenizer(source_texts, padding="max_length", truncation=True, max_length=128)
    target = tokenizer(target_texts, padding="max_length", truncation=True, max_length=128)

    return {
        'input_ids': source['input_ids'],
        'attention_mask': source['attention_mask'],
        'labels': target['input_ids']
    }

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          # Where to save model checkpoints
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=44, # Batch size per GPU
    per_device_eval_batch_size=44,  # Batch size for validation
    num_train_epochs=5,             # Number of epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_total_limit=3,
    report_to="none" # Keep only the last 3 checkpoints
)

In [ ]:
from transformers import Trainer
import os

# Create the trainer
trainer = Trainer(
    model=new_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

# Start training
trainer.train()

# 9. Save
# Save the model and tokenizer to Google Drive
model.save_pretrained('/content/drive/MyDrive/fine_tuned_nmt_model2')
tokenizer.save_pretrained('/content/drive/MyDrive/fine_tuned_nmt_model2')